In [2]:
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv('application_train.csv')
print('Shape:', df.shape)
print('\nTarget:')
print(df['TARGET'].value_counts(normalize=True))
print('\nПропуски топ-20:')
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(missing[missing > 0].head(20))
print('\nЧисло категориальных фич:')
print(df.select_dtypes(include='object').shape[1])
print('\nЧисло числовых фич:')
print(df.select_dtypes(include='number').shape[1])

Shape: (307511, 122)

Target:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

Пропуски топ-20:
COMMONAREA_MEDI             69.872297
COMMONAREA_AVG              69.872297
COMMONAREA_MODE             69.872297
NONLIVINGAPARTMENTS_MODE    69.432963
NONLIVINGAPARTMENTS_AVG     69.432963
NONLIVINGAPARTMENTS_MEDI    69.432963
FONDKAPREMONT_MODE          68.386172
LIVINGAPARTMENTS_MODE       68.354953
LIVINGAPARTMENTS_AVG        68.354953
LIVINGAPARTMENTS_MEDI       68.354953
FLOORSMIN_AVG               67.848630
FLOORSMIN_MODE              67.848630
FLOORSMIN_MEDI              67.848630
YEARS_BUILD_MEDI            66.497784
YEARS_BUILD_MODE            66.497784
YEARS_BUILD_AVG             66.497784
OWN_CAR_AGE                 65.990810
LANDAREA_MEDI               59.376738
LANDAREA_MODE               59.376738
LANDAREA_AVG                59.376738
dtype: float64

Число категориальных фич:
16

Число числовых фич:
106


Группы фичей с большими пропусками + категориальные и наиболее коррелирующие фичи \
Порог, за который мы принимаем наиболее пропущенных фичей поставим 40 процентов

In [4]:
high_missing = missing[missing > 40].index.tolist()
print(f'Фичи с пропусками >40%: {len(high_missing)}')

# Категориальные
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'\nКатегориальные фичи:')
for col in cat_cols:
    print(f'  {col}: {df[col].nunique()} уникальных')

# Топ числовых по корреляции с таргетом
num_corr = df.select_dtypes(include='number').corr()['TARGET'].abs().sort_values(ascending=False)
print(f'\nТоп-15 числовых по корреляции с таргетом:')
print(num_corr.head(15))

Фичи с пропусками >40%: 49

Категориальные фичи:
  NAME_CONTRACT_TYPE: 2 уникальных
  CODE_GENDER: 3 уникальных
  FLAG_OWN_CAR: 2 уникальных
  FLAG_OWN_REALTY: 2 уникальных
  NAME_TYPE_SUITE: 7 уникальных
  NAME_INCOME_TYPE: 8 уникальных
  NAME_EDUCATION_TYPE: 5 уникальных
  NAME_FAMILY_STATUS: 6 уникальных
  NAME_HOUSING_TYPE: 6 уникальных
  OCCUPATION_TYPE: 18 уникальных
  WEEKDAY_APPR_PROCESS_START: 7 уникальных
  ORGANIZATION_TYPE: 58 уникальных
  FONDKAPREMONT_MODE: 4 уникальных
  HOUSETYPE_MODE: 3 уникальных
  WALLSMATERIAL_MODE: 7 уникальных
  EMERGENCYSTATE_MODE: 2 уникальных

Топ-15 числовых по корреляции с таргетом:
TARGET                         1.000000
EXT_SOURCE_3                   0.178919
EXT_SOURCE_2                   0.160472
EXT_SOURCE_1                   0.155317
DAYS_BIRTH                     0.078239
REGION_RATING_CLIENT_W_CITY    0.060893
REGION_RATING_CLIENT           0.058899
DAYS_LAST_PHONE_CHANGE         0.055218
DAYS_ID_PUBLISH                0.051457
REG_CI

In [5]:
high_missing

['COMMONAREA_MEDI',
 'COMMONAREA_AVG',
 'COMMONAREA_MODE',
 'NONLIVINGAPARTMENTS_MODE',
 'NONLIVINGAPARTMENTS_AVG',
 'NONLIVINGAPARTMENTS_MEDI',
 'FONDKAPREMONT_MODE',
 'LIVINGAPARTMENTS_MODE',
 'LIVINGAPARTMENTS_AVG',
 'LIVINGAPARTMENTS_MEDI',
 'FLOORSMIN_AVG',
 'FLOORSMIN_MODE',
 'FLOORSMIN_MEDI',
 'YEARS_BUILD_MEDI',
 'YEARS_BUILD_MODE',
 'YEARS_BUILD_AVG',
 'OWN_CAR_AGE',
 'LANDAREA_MEDI',
 'LANDAREA_MODE',
 'LANDAREA_AVG',
 'BASEMENTAREA_MEDI',
 'BASEMENTAREA_AVG',
 'BASEMENTAREA_MODE',
 'EXT_SOURCE_1',
 'NONLIVINGAREA_MODE',
 'NONLIVINGAREA_AVG',
 'NONLIVINGAREA_MEDI',
 'ELEVATORS_MEDI',
 'ELEVATORS_AVG',
 'ELEVATORS_MODE',
 'WALLSMATERIAL_MODE',
 'APARTMENTS_MEDI',
 'APARTMENTS_AVG',
 'APARTMENTS_MODE',
 'ENTRANCES_MEDI',
 'ENTRANCES_AVG',
 'ENTRANCES_MODE',
 'LIVINGAREA_AVG',
 'LIVINGAREA_MODE',
 'LIVINGAREA_MEDI',
 'HOUSETYPE_MODE',
 'FLOORSMAX_MODE',
 'FLOORSMAX_MEDI',
 'FLOORSMAX_AVG',
 'YEARS_BEGINEXPLUATATION_MODE',
 'YEARS_BEGINEXPLUATATION_MEDI',
 'YEARS_BEGINEXPLUATATIO

Пропуски — 49 фич с пропусками >40%, это в основном фичи про недвижимость (площади, этажи, год постройки). Их проще всего дропнуть или заполнить медианой. \
Категориальные — 16 фич, все осмысленные: пол, тип контракта, образование, занятость, тип жилья. \
ORGANIZATION_TYPE самая богатая — 58 уникальных значений.
Самые важные числовые фичи:

EXT_SOURCE_1/2/3 — внешние скоринговые оценки, корреляция 0.16-0.18. Это самые сильные фичи в датасете — аналог TotalLatePayments_weighted в Give Me Credit \
DAYS_BIRTH — возраст (отрицательный, дней от рождения) \
DAYS_EMPLOYED — стаж работы \
Остальные корреляции слабые (< 0.06) 

Соответсвенно, дропаем все фичи кроме EXT_SOURCE_1 - так как опали в топ коррелирующих фичей

In [7]:
drop_cols = missing[(missing > 40) & (missing.index != 'EXT_SOURCE_1')].index.tolist()

In [8]:
drop_but_correlated = [c for c in drop_cols if c in num_corr.head(20).index]
print(drop_but_correlated)

['OWN_CAR_AGE', 'FLOORSMAX_MODE', 'FLOORSMAX_MEDI', 'FLOORSMAX_AVG']


num_corr.head(20)

In [9]:
num_corr.head(20)

TARGET                         1.000000
EXT_SOURCE_3                   0.178919
EXT_SOURCE_2                   0.160472
EXT_SOURCE_1                   0.155317
DAYS_BIRTH                     0.078239
REGION_RATING_CLIENT_W_CITY    0.060893
REGION_RATING_CLIENT           0.058899
DAYS_LAST_PHONE_CHANGE         0.055218
DAYS_ID_PUBLISH                0.051457
REG_CITY_NOT_WORK_CITY         0.050994
FLAG_EMP_PHONE                 0.045982
DAYS_EMPLOYED                  0.044932
REG_CITY_NOT_LIVE_CITY         0.044395
FLAG_DOCUMENT_3                0.044346
FLOORSMAX_AVG                  0.044003
FLOORSMAX_MEDI                 0.043768
FLOORSMAX_MODE                 0.043226
DAYS_REGISTRATION              0.041975
AMT_GOODS_PRICE                0.039645
OWN_CAR_AGE                    0.037612
Name: TARGET, dtype: float64

In [13]:
df["FLAG_OWN_CAR"].unique()

array(['N', 'Y'], dtype=object)

In [14]:
missing_pct = df.isnull().sum() / len(df) * 100
drop_cols = missing[
    (missing > 40) & (missing.index != 'EXT_SOURCE_1')
].index.tolist()

df.drop(columns=drop_cols, inplace=True)
print(f'Дропнуто: {len(drop_cols)} фич → shape: {df.shape}')

Дропнуто: 48 фич → shape: (307511, 74)


In [15]:
df.info

<bound method DataFrame.info of         SK_ID_CURR  TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR  \
0           100002       1         Cash loans           M            N   
1           100003       0         Cash loans           F            N   
2           100004       0    Revolving loans           M            Y   
3           100006       0         Cash loans           F            N   
4           100007       0         Cash loans           M            N   
...            ...     ...                ...         ...          ...   
307506      456251       0         Cash loans           M            N   
307507      456252       0         Cash loans           F            N   
307508      456253       0         Cash loans           F            N   
307509      456254       1         Cash loans           F            N   
307510      456255       0         Cash loans           F            N   

       FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  \
0        

In [17]:
print('=== ЧИСЛОВЫЕ ФИЧИ ===\n')
num_stats = df.select_dtypes(include='number').describe().T
num_stats['missing_%'] = (df.select_dtypes(include='number').isnull().sum() / len(df) * 100).round(2)
num_stats = num_stats[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'missing_%']]
print(num_stats.to_string())

=== ЧИСЛОВЫЕ ФИЧИ ===

                                count           mean            std           min            25%            50%            75%           max  missing_%
SK_ID_CURR                   307511.0  278180.518577  102790.175348  1.000020e+05  189145.500000  278202.000000  367142.500000  4.562550e+05       0.00
TARGET                       307511.0       0.080729       0.272419  0.000000e+00       0.000000       0.000000       0.000000  1.000000e+00       0.00
CNT_CHILDREN                 307511.0       0.417052       0.722121  0.000000e+00       0.000000       0.000000       1.000000  1.900000e+01       0.00
AMT_INCOME_TOTAL             307511.0  168797.919297  237123.146279  2.565000e+04  112500.000000  147150.000000  202500.000000  1.170000e+08       0.00
AMT_CREDIT                   307511.0  599025.999706  402490.776996  4.500000e+04  270000.000000  513531.000000  808650.000000  4.050000e+06       0.00
AMT_ANNUITY                  307499.0   27108.573909   14493.7373

In [18]:
print('\n=== КАТЕГОРИАЛЬНЫЕ ФИЧИ ===\n')
for col in df.select_dtypes(include='object').columns:
    missing = df[col].isnull().sum() / len(df) * 100
    print(f'{col}')
    print(f'  Уникальных: {df[col].nunique()} | Пропуски: {missing:.2f}%')
    print(f'  Значения: {df[col].value_counts().to_dict()}')
    print()


=== КАТЕГОРИАЛЬНЫЕ ФИЧИ ===

NAME_CONTRACT_TYPE
  Уникальных: 2 | Пропуски: 0.00%
  Значения: {'Cash loans': 278232, 'Revolving loans': 29279}

CODE_GENDER
  Уникальных: 3 | Пропуски: 0.00%
  Значения: {'F': 202448, 'M': 105059, 'XNA': 4}

FLAG_OWN_CAR
  Уникальных: 2 | Пропуски: 0.00%
  Значения: {'N': 202924, 'Y': 104587}

FLAG_OWN_REALTY
  Уникальных: 2 | Пропуски: 0.00%
  Значения: {'Y': 213312, 'N': 94199}

NAME_TYPE_SUITE
  Уникальных: 7 | Пропуски: 0.42%
  Значения: {'Unaccompanied': 248526, 'Family': 40149, 'Spouse, partner': 11370, 'Children': 3267, 'Other_B': 1770, 'Other_A': 866, 'Group of people': 271}

NAME_INCOME_TYPE
  Уникальных: 8 | Пропуски: 0.00%
  Значения: {'Working': 158774, 'Commercial associate': 71617, 'Pensioner': 55362, 'State servant': 21703, 'Unemployed': 22, 'Student': 18, 'Businessman': 10, 'Maternity leave': 5}

NAME_EDUCATION_TYPE
  Уникальных: 5 | Пропуски: 0.00%
  Значения: {'Secondary / secondary special': 218391, 'Higher education': 74863, 'Incompl

1. SK_ID_CURR — дропаем, это просто ID клиента, не фича.
2. Почти константные фичи — дропаем, они не несут информации:
Фича  Mean  Что это значит \
FLAG_MOBIL 0.999997 у всех есть мобильный \
FLAG_CONT_MOBILE 0.998133 почти у всех \
4. Редкие FLAG_DOCUMENT — дропаем, встречаются в <0.1% записей:
FLAG_DOCUMENT_2, 4, 7, 10, 12, 17, 19, 20, 21
5. DAYS_EMPLOYED max = 365243 — аномалия ещё не исправлена 
6. Выбросы:

AMT_INCOME_TOTAL max = 117M при медиане 147K — явный выброс \
AMT_REQ_CREDIT_BUREAU_QRT max = 261 при медиане 0 \
CNT_CHILDREN max = 19, CNT_FAM_MEMBERS max = 20 

In [19]:
df.drop(columns=['SK_ID_CURR'], inplace=True)

# Дропаем почти константные
df.drop(columns=['FLAG_MOBIL', 'FLAG_CONT_MOBILE'], inplace=True)

# Дропаем редкие документы
rare_docs = ['FLAG_DOCUMENT_2', 'FLAG_DOCUMENT_4', 'FLAG_DOCUMENT_7',
             'FLAG_DOCUMENT_10', 'FLAG_DOCUMENT_12', 'FLAG_DOCUMENT_17',
             'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20', 'FLAG_DOCUMENT_21']
df.drop(columns=rare_docs, inplace=True)

print('Shape после дропа констант и редких:', df.shape)

Shape после дропа констант и редких: (307511, 62)


CODE_GENDER — XNA (4 записи) - можно дропать

In [20]:
df = df[df['CODE_GENDER'] != 'XNA']
print('Shape после удаления XNA:', df.shape)  # 307507

Shape после удаления XNA: (307507, 62)


NAME_FAMILY_STATUS — есть Unknown у 2 записей. тоже дропаем:

In [21]:
df = df[df['NAME_FAMILY_STATUS'] != 'Unknown']
print('Shape после удаления Unknown:', df.shape)

Shape после удаления Unknown: (307505, 62)


NAME_INCOME_TYPE — очень редкие категории:

Unemployed — 22, Student — 18, Businessman — 10, Maternity leave — 5
Объединяем в Other:

In [22]:
rare_income = ['Unemployed', 'Student', 'Businessman', 'Maternity leave']
df['NAME_INCOME_TYPE'] = df['NAME_INCOME_TYPE'].replace(rare_income, 'Other')

OCCUPATION_TYPE — 31% пропусков. Это не случайные пропуски — скорее всего незаполненная анкета. Заполняем отдельной категорией Unknown:

In [23]:
df['OCCUPATION_TYPE'] = df['OCCUPATION_TYPE'].fillna('Unknown')

ORGANIZATION_TYPE:

XNA (55374 записей) — не пропуск, но фактически "неизвестно". Переименуем в Unknown для ясности

In [24]:
df['ORGANIZATION_TYPE'] = df['ORGANIZATION_TYPE'].replace('XNA', 'Unknown')

In [27]:
# другие фичи заполняем медианой
num_cols = [c for c in df.select_dtypes(include='number').columns if c != 'TARGET']
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

In [31]:
print(f'Shape: {df.shape}')
print(f'NaN: {df.isnull().sum().sum()}')

Shape: (307505, 62)
NaN: 0


In [28]:
print(df.isnull().sum()[df.isnull().sum() > 0])

NAME_TYPE_SUITE    1290
dtype: int64


In [29]:
df["NAME_TYPE_SUITE"].mode()

0    Unaccompanied
Name: NAME_TYPE_SUITE, dtype: object

In [30]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [32]:
df

,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_18,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,297000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
4,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,513000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,225000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
307507,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,225000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
307508,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,585000.0,...,0,0,0,0,1.0,0.0,0.0,1.0,0.0,1.0
307509,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,319500.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [33]:
df.to_csv("application_train_all_models.csv")